# Notebook 13: Clustering & High Availability

So far, we've used a **single Redis server**. In production, you need:

- **High Availability (HA):** What if the server crashes? → **Redis Sentinel**
- **Horizontal Scaling:** What if one server's memory isn't enough? → **Redis Cluster**

This notebook covers the concepts, architecture, and Python connection patterns for both.

> Note: We can't run a full Sentinel/Cluster setup in a single Docker container, so some examples are shown as code patterns rather than executable demos. The executable sections cover connection pooling, monitoring, and security — all of which work on our single instance.

In [ ]:
import redis
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
print(f"Connected! Redis version: {r.info('server')['redis_version']}")

---
## Part 1: Replication

Before understanding Sentinel or Cluster, you need to understand **replication**.

```
┌──────────┐     replication      ┌──────────┐
│  Master   │ ──────────────────► │  Replica  │
│ (writes)  │                     │ (reads)   │
└──────────┘                     └──────────┘
                                  ┌──────────┐
              ──────────────────► │  Replica  │
                                  │ (reads)   │
                                  └──────────┘
```

- **Master** accepts writes and reads
- **Replicas** are read-only copies of the master
- Replication is **asynchronous** by default
- If the master dies, a replica can be promoted

In [ ]:
# Check replication info on our server
info = r.info('replication')
print("Replication info:")
print(f"  Role: {info['role']}")
print(f"  Connected replicas: {info['connected_slaves']}")

# In a real setup with replicas, you'd see:
# role: master
# connected_slaves: 2
# slave0: ip=..., port=..., state=online

---
## Part 2: Redis Sentinel — High Availability

**Problem:** If the master crashes, your app is down.

**Solution:** Sentinel monitors Redis instances and performs **automatic failover**.

```
┌──────────┐   ┌──────────┐   ┌──────────┐
│ Sentinel │   │ Sentinel │   │ Sentinel │
│    1     │   │    2     │   │    3     │
└────┬─────┘   └────┬─────┘   └────┬─────┘
     │              │              │
     └──── monitoring ────────────┘
                    │
          ┌─────────┼─────────┐
          ▼         ▼         ▼
     ┌────────┐ ┌────────┐ ┌────────┐
     │ Master │ │Replica │ │Replica │
     │  :6379 │ │  :6380 │ │  :6381 │
     └────────┘ └────────┘ └────────┘
```

### How Failover Works
1. Sentinels continuously ping the master
2. If the master doesn't respond, sentinels **vote** on whether it's truly down
3. If the majority agrees, one sentinel **promotes a replica** to master
4. Other replicas are reconfigured to follow the new master
5. When the old master comes back, it becomes a replica

### Sentinel Guarantees
- At least 3 sentinels (for majority voting)
- Automatic failover in seconds
- Clients are notified of the new master

In [ ]:
# Connecting via Sentinel in Python
# (This is the pattern — requires actual sentinel setup)

print("""Sentinel Connection Pattern:

from redis.sentinel import Sentinel

# Connect to sentinel instances
sentinel = Sentinel([
    ('sentinel-host-1', 26379),
    ('sentinel-host-2', 26379),
    ('sentinel-host-3', 26379)
], socket_timeout=0.1)

# Get a connection to the MASTER (for writes)
master = sentinel.master_for('mymaster', socket_timeout=0.1, decode_responses=True)
master.set('key', 'value')  # Writes go to master

# Get a connection to a REPLICA (for reads)
slave = sentinel.slave_for('mymaster', socket_timeout=0.1, decode_responses=True)
value = slave.get('key')  # Reads from replica

# If master fails, sentinel automatically redirects your connection!
""")

---
## Part 3: Redis Cluster — Horizontal Scaling

**Problem:** Your data is too big for one server (e.g., 200GB of data, but servers have 64GB RAM).

**Solution:** Redis Cluster **shards** data across multiple masters.

```
┌──────────────────────────────────────────────┐
│           16384 Hash Slots                    │
│  [0────5460]  [5461───10922]  [10923───16383] │
│       │              │              │         │
│  ┌────┴────┐   ┌────┴────┐   ┌────┴────┐    │
│  │ Master A │   │ Master B │   │ Master C │    │
│  │ + Replica│   │ + Replica│   │ + Replica│    │
│  └─────────┘   └─────────┘   └─────────┘    │
└──────────────────────────────────────────────┘
```

### How It Works
1. The keyspace is divided into **16384 hash slots**
2. Each master is responsible for a subset of slots
3. Key → `CRC16(key) % 16384` → slot → node
4. Client connects to the right node automatically

### Hash Tags
By default, each key might go to a different node. If you need related keys on the same node, use **hash tags**:

```
{user:1001}.name  → slot determined by 'user:1001'
{user:1001}.email → same slot! (same hash tag)
```

This ensures related keys are on the same node, enabling multi-key operations.

In [ ]:
# Redis Cluster connection pattern
print("""Cluster Connection Pattern:

from redis.cluster import RedisCluster

# Connect to the cluster (just need one node — it discovers the rest)
rc = RedisCluster(
    host='cluster-node-1',
    port=7000,
    decode_responses=True
)

# Use it like a normal Redis connection!
rc.set('user:1001', 'Sujit')
value = rc.get('user:1001')

# The client automatically routes to the correct node.

# Hash tags for multi-key operations:
rc.set('{user:1001}.name', 'Sujit')     # Same slot
rc.set('{user:1001}.email', 'a@b.com')  # Same slot
rc.mget('{user:1001}.name', '{user:1001}.email')  # Works!
""")

# Demonstrate hash slot calculation locally
from binascii import crc_hqx

def calculate_slot(key):
    """Calculate which hash slot a key belongs to."""
    # Check for hash tag
    start = key.find('{')
    if start != -1:
        end = key.find('}', start + 1)
        if end != -1 and end != start + 1:
            key = key[start+1:end]
    return crc_hqx(key.encode(), 0) % 16384

# Different keys, different slots
print("\nHash slot distribution:")
for key in ['user:1001', 'user:1002', 'product:1', 'session:abc']:
    print(f"  {key:20s} → slot {calculate_slot(key)}")

# Hash tags force same slot
print("\nWith hash tags (same slot):")
for key in ['{user:1001}.name', '{user:1001}.email', '{user:1001}.age']:
    print(f"  {key:25s} → slot {calculate_slot(key)}")

### Cluster Limitations

| Limitation | Details |
|---|---|
| Multi-key ops | Only work if keys are on the same slot (use hash tags) |
| SELECT | Only database 0 is available |
| Lua scripts | All keys in a script must be on the same slot |
| Transactions | MULTI/EXEC only within a single slot |
| Min nodes | 3 masters + 3 replicas = 6 nodes minimum |

---
## Sentinel vs Cluster

| Feature | Sentinel | Cluster |
|---|---|---|
| Purpose | High availability (failover) | Scaling (sharding) |
| Data distribution | All data on one master | Data split across masters |
| Max dataset size | Single server's RAM | Sum of all masters' RAM |
| Write scaling | No (single master) | Yes (multiple masters) |
| Read scaling | Yes (read from replicas) | Yes |
| Failover | Automatic | Automatic |
| Complexity | Moderate | Higher |
| Min nodes | 1 master + 2 replicas + 3 sentinels | 6 (3 masters + 3 replicas) |
| When to use | Data fits in one server | Data too big for one server |

---
## Part 4: Connection Pooling

Creating a new connection for every operation is expensive. **Connection pools** reuse connections.

In [ ]:
# Connection pool — reuse connections instead of creating new ones
pool = redis.ConnectionPool(
    host='localhost',
    port=6379,
    db=0,
    max_connections=10,       # Max 10 simultaneous connections
    decode_responses=True
)

# Create Redis client using the pool
r_pooled = redis.Redis(connection_pool=pool)

# Use it normally — connections are managed automatically
r_pooled.set('pooled', 'hello')
print(f"Value: {r_pooled.get('pooled')}")

# Pool info
print(f"\nPool size: {pool.max_connections}")
print(f"Pool class: {type(pool).__name__}")
print("\nIn production, always use connection pooling!")

---
## Part 5: Security

In [ ]:
# Security configuration options
print("=== Redis Security Checklist ===")
print()
print("1. SET A PASSWORD:")
print("   # redis.conf: requirepass YourStrongPassword")
print("   # Python: redis.Redis(password='YourStrongPassword')")
print()
print("2. BIND TO SPECIFIC IPs:")
print("   # redis.conf: bind 127.0.0.1 192.168.1.100")
print("   # Never bind to 0.0.0.0 in production!")
print()
print("3. USE ACLs (Redis 6+):")
print("   # Create users with limited permissions")
print("   # ACL SETUSER readonly ~cached:* +get +mget +keys")
print()
print("4. ENABLE TLS:")
print("   # redis.conf: tls-port 6380")
print("   # Python: redis.Redis(ssl=True, ssl_certfile='cert.pem')")
print()
print("5. DISABLE DANGEROUS COMMANDS:")
print("   # redis.conf: rename-command FLUSHALL ''")
print("   # Prevents accidental data deletion")

In [ ]:
# Check ACL configuration (Redis 6+)
try:
    acl_list = r.acl_list()
    print("ACL Users:")
    for acl in acl_list:
        print(f"  {acl}")
except redis.ResponseError as e:
    print(f"ACL info: {e}")

---
## Part 6: Monitoring

In [ ]:
# Essential monitoring commands

# 1. Server info
server = r.info('server')
print(f"Redis version: {server['redis_version']}")
print(f"Uptime: {server['uptime_in_seconds']}s")

# 2. Memory
memory = r.info('memory')
print(f"\nUsed memory: {memory['used_memory_human']}")
print(f"Peak memory: {memory['used_memory_peak_human']}")

# 3. Clients
clients = r.info('clients')
print(f"\nConnected clients: {clients['connected_clients']}")

# 4. Stats
stats = r.info('stats')
print(f"\nTotal commands processed: {stats['total_commands_processed']}")
print(f"Keyspace hits: {stats['keyspace_hits']}")
print(f"Keyspace misses: {stats['keyspace_misses']}")

# 5. Database size
print(f"\nTotal keys: {r.dbsize()}")

In [ ]:
# Slow log — find slow commands
# Redis CLI: SLOWLOG GET 5
slowlog = r.slowlog_get(5)
print(f"Slow log entries: {len(slowlog)}")
for entry in slowlog:
    print(f"  Command: {entry.get('command', 'N/A')}, Duration: {entry.get('duration', 'N/A')}us")

In [ ]:
# Client list — see who's connected
client_list = r.client_list()
print(f"Connected clients ({len(client_list)}):")
for client in client_list[:5]:  # Show first 5
    print(f"  addr={client['addr']}, age={client['age']}s, db={client['db']}")

---
## Part 7: Redis in Production — Checklist

### Before Going Live

| Category | Action |
|---|---|
| **Memory** | Set `maxmemory` and `maxmemory-policy` |
| **Persistence** | Choose RDB, AOF, or both based on your needs |
| **Security** | Set password, bind to specific IPs, use TLS |
| **Connections** | Use connection pooling in your app |
| **Monitoring** | Monitor memory, connections, hit ratio, slow log |
| **Backup** | Regular RDB backups to external storage |
| **HA** | Use Sentinel or Cluster for failover |
| **Keys** | Always set TTL on cache keys |
| **Scanning** | Never use KEYS in production, use SCAN |
| **Logging** | Monitor slow log for performance issues |

---
## Course Recap — Everything You've Learned!

| Notebook | Topic | Key Concepts |
|---|---|---|
| 01 | Hello Redis | What Redis is, SET/GET, key-value basics |
| 02 | Strings | INCR, expiry, NX/XX, bit operations, rate limiting |
| 03 | Lists | LPUSH/RPUSH, queues, stacks, activity feeds |
| 04 | Sets & Sorted Sets | Unique collections, set operations, leaderboards |
| 05 | Hashes | Object storage, CRUD operations, shopping carts |
| 06 | Key Expiry & TTL | TTL, EXPIRE, SCAN, sessions, OTP codes |
| 07 | Pub/Sub | Real-time messaging, channels, patterns |
| 08 | Transactions | MULTI/EXEC, Pipelines, WATCH, optimistic locking |
| 09 | Persistence | RDB snapshots, AOF logging |
| 10 | Caching Patterns | Cache-aside, write-through, eviction, stampede |
| 11 | Streams | Append-only logs, consumer groups, XADD/XREAD |
| 12 | Lua Scripting | Custom atomic operations, register_script |
| 13 | Cluster & Sentinel | High availability, sharding, production ops |

You now have a **solid understanding** of Redis from basics to production-ready concepts!

---
## What's Next?

### Redis Stack (Extended Modules)
- **RedisJSON** — Native JSON document storage and querying
- **RediSearch** — Full-text search engine built into Redis
- **RedisTimeSeries** — Time-series data with downsampling
- **RedisGraph** — Graph database functionality
- **RedisBloom** — Probabilistic data structures (Bloom filters, etc.)

### Practice Project Ideas
1. **URL Shortener** — Use Redis for mapping short codes to URLs
2. **Real-Time Chat App** — Pub/Sub + Streams + Lists for history
3. **API Rate Limiter Middleware** — For a Flask/FastAPI app
4. **Job Queue System** — Reliable queue with retry and dead-letter handling
5. **Session-Based Auth** — Login/logout with Redis session storage
6. **Leaderboard Service** — Complete gaming leaderboard API
7. **Caching Layer** — Add Redis caching to an existing database app

---
## Exercises

1. **Health Check Dashboard:** Write a function that collects and displays: Redis version, uptime, memory usage, connected clients, total keys, hit ratio, and slow queries.

2. **Connection Pool Test:** Create a connection pool with max 5 connections. Launch 10 threads that all try to use Redis simultaneously. Observe how the pool manages connections.

3. **Hash Slot Calculator:** Write a function that takes a list of keys and shows which cluster node each would go to (assume 3 masters with slot ranges 0-5460, 5461-10922, 10923-16383).

4. **Production Config Generator:** Write a function that generates a recommended Redis configuration based on use case (cache/database/queue), available memory, and required durability.

In [ ]:
# Your exercises here!


---
### Congratulations!

You've completed the entire Redis learning series — from your first `SET`/`GET` to understanding production clustering. You now have the knowledge to use Redis effectively in real-world applications.

Happy coding!